<a href="https://colab.research.google.com/github/Kanishaag/AI-Fashion-Recommendation-Engine/blob/main/Fashion_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*AI Fashion Recommendation Engine using NLP, Transformers, and Semantic Search*

1. Installing and Importing Dataset Library


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

2. Loading the Research Paper Dataset

In [ ]:
dataset = load_dataset("Censius-AI/ECommerce-Women-Clothing-Reviews", split = "train")

In [ ]:
print(dataset)

In [ ]:
dataset[0]

3. Converting Dataset into Pandas DataFrame

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(dataset)
df

In [ ]:
df.columns

4. Selecting Required Columns

In [ ]:
df = df[["Title", "Review Text"]].copy()

In [ ]:
df.shape

5. Checking Missing Values

In [ ]:
df.isnull().sum()

6. Creating & Displaying  Combined Text Column

In [ ]:
df["paper_text"] = df["Title"] + " " + df["Review Text"]

In [ ]:
df[["paper_text"]].head()

In [ ]:
print(df["paper_text"].iloc[0])

7. Reducing Dataset Size

In [ ]:
df = df.dropna(subset=["paper_text"])

In [ ]:
df = df.head(10000)
df

8. Loading Sentence Transformer Model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
print(type(model))

In [ ]:
print(type(df["paper_text"].iloc[0]))

9. Cleaning Text

In [ ]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex = False)
df["paper_text"] = df["paper_text"].str.strip()

In [ ]:
sample_text = df["paper_text"].iloc[0]
sample_text

10. Generating Embeddings

In [ ]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

In [ ]:
embedding[:64]

11. Creating Embeddings for Multiple Reviews

In [ ]:
sample_embedding = model.encode(df["paper_text"].head(5).to_list())

In [ ]:
print(sample_embedding.shape)

12. Calculating Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1),
                            sample_embedding[0].reshape(1, -1))
print(similarity)

In [ ]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1),
                               sample_embedding[1].reshape(1, -1))
print(similarity)

In [ ]:
for i in  range(1, 5):
  sim = cosine_similarity(sample_embedding[0].reshape(1, -1),
                          sample_embedding[i].reshape(1, -1))
  print(sim)

13. Saving Embeddings

In [ ]:
import os
import numpy as np

if os.path.exists("fashion_embeddings.npy"):
    print("Loading saved fashion embeddings...")
    embeddings = np.load("fashion_embeddings.npy")
else:
    print("Generating brand-new fashion embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("fashion_embeddings.npy", embeddings)
    print("Fashion embeddings saved successfully!")

In [ ]:
print(embeddings.shape)

In [ ]:
print(type(embedding))

In [ ]:
embedding.dtype

14. Installing and Importing FAISS

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

15. Normalizing Embeddings

In [ ]:
faiss.normalize_L2(embeddings)

16. Creating FAISS Index

In [ ]:
index = faiss.IndexFlatIP(384)

17. Adding Embeddings to Index

In [ ]:
index.add(embeddings)

In [ ]:
if os.path.exists("fashion_embeddings.index"):
    print("Loading existing Fashion FAISS index")
    index = faiss.read_index("fashion_embeddings.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "fashion_embeddings.index")
    print("Fashion FAISS index saved successfully!")

In [ ]:
print(f"Total fashion items indexed: {index.ntotal}")

In [ ]:
query = "lightweight and breathable dress for hot summer beach days"

In [ ]:
query_embedding = model.encode([query])
print(query_embedding.shape)

18. Creating the Search Function


In [ ]:
def search_fashion_item(query, k = 5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    return D, I

19. Searching for Similar Review

In [ ]:
D, I = search_fashion_item(query, 5)
print("Similarity Scores:\n", D)
print("Row Indexes:\n", I)

In [ ]:
for score, idx in zip(D[0], I[0]):
    print(f"Similarity Score: {score:.2f}")
    print(f"Product Title: {df.iloc[idx]['Title']}")
    print(f"Review Text: {df.iloc[idx]['Review Text']}")
    print("-" * 50 + "\n")

20. Installing Transformers

In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline

21. Initializing BART Summarization Pipeline

In [ ]:
summarizer = pipeline(
    "summarization",
    model = "facebook/bart-large-cnn"

)

In [ ]:
type(summarizer)

22. Generating Summary


In [ ]:
summary = summarizer(df.iloc[642]["Review Text"], max_length = 64, min_length = 32)
print(summary)

In [ ]:
type(summary)

In [ ]:
type(summary[0])

In [ ]:
summary[0]["summary_text"]

23. Creating the Final Search and Summarize Function

In [ ]:
def search_and_summarize_fashion(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    print("="*60)
    print(f"RECOMMENDED OUTFITS FOR: '{query.upper()}'")
    print("="*60 + "\n")

    for score, idx in zip(D[0], I[0]):
        review_title = df.iloc[idx]["Title"]
        full_review = df.iloc[idx]["Review Text"]

        print("Similarity Score:", round(float(score), 2))
        print("Review Title:", review_title)

        summary = summarizer(
            full_review,
            max_length=50,
            min_length=20,
            do_sample=False
        )

        print("AI Summary of Reviews:", summary[0]["summary_text"])
        print("-" * 50 + "\n")

In [ ]:
search_and_summarize_fashion("cute cover-up or summer top & shorts", k=3)

24. Installing KeyBERT

In [ ]:
!pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

25. Initializing KeyBERT

In [ ]:
kw_model = KeyBERT()

In [ ]:
type(kw_model)

26. Keyword Mining

In [ ]:
text = df.iloc[106]["Review Text"]
keywords = kw_model.extract_keywords(text)
print(keywords)

In [ ]:
print(type(keywords))

In [ ]:
print(type(keywords[0]))

27. End-to-End Search, Summarization and Tagging Pipeline

In [ ]:
def deep_fashion_search(query, k=3):
    # 1. AI Vector Search
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    print("="*60)
    print(f"AI ENGINE RESULTS FOR: '{query.upper()}'")
    print("="*60 + "\n")

    # 2. Extract, Summarize, and Tag Data
    for score, idx in zip(D[0], I[0]):
        title = df.iloc[idx]["Title"]
        review = df.iloc[idx]["Review Text"]

        print(f"Similarity Score: {round(float(score), 2)}")
        print(f"Review Title: {title}")

        # Summary Generation
        summary = summarizer(review, max_length=50, min_length=20, do_sample=False)
        print(f"AI Summary: {summary[0]['summary_text']}")

        # Keyphrase Mining
        keywords = kw_model.extract_keywords(
            review,
            keyphrase_ngram_range=(1, 3),
            stop_words="english",
            top_n=3
        )

        # Format and print the keywords
        extracted_tags = [kw[0] for kw in keywords]
        print(f"Extracted Fashion Tags: {', '.join(extracted_tags)}")
        print("-" * 50 + "\n")

In [ ]:
deep_fashion_search("cute cover-up or summer top & shorts")

28. Implementing Named Entity Recognition (NER) Processes

#Approach 1: Hugging Face Transformers for Feature Extraction (BERT for NER)

In [ ]:
from transformers import pipeline

In [ ]:
hf_ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

In [ ]:
def run_approach_one_ner(text_to_test):
    print("-" * 50)
    print("OUTPUT FOR APPROACH 1: HUGGING FACE BERT-NER")

    results = hf_ner_pipeline(str(text_to_test))

    if not results:
        print("No standard entities found.")
    else:
        for entity in results:
            print(f"Found Entity: '{entity['word']}' ---> Label: {entity['entity_group']} (Confidence: {round(entity['score'], 2)})")

In [ ]:
sample_text_1 = "I wore this amazing sundress on my vacation to Paris and California!"
run_approach_one_ner(sample_text_1)

# Approach 2: Custom Dictionary-Based Fashion-NER

In [ ]:
import re

In [ ]:
def run_approach_two_ner(text_to_test):
    print("-" * 50)
    print("OUTPUT FOR APPROACH 2: CUSTOM DICTIONARY FASHION-NER")

    fashion_dictionary = {
        "Fabric": r"\b(cotton|linen|silk|denim|leather|lace|knit|wool|polyester|rayon|blend|satin)\b",
        "Size & Style": r"\b(petite|regular|plus size|oversized|true to size|small|medium|large|xs|xl|xxl|short|long|loose|tight)\b",
        "Categories": r"\b(dress|top|shorts|pants|skirt|blouse|jeans|jacket|sweater|maxi|cover-up|romper|jumpsuit|tee|shirt)\b"
    }

    found_any = False
    for category, regex_pattern in fashion_dictionary.items():
        matches = set(re.findall(regex_pattern, str(text_to_test), flags=re.IGNORECASE))
        if matches:
            found_any = True
            print(f"{category} Detected: {', '.join(matches)}")

    if not found_any:
        print("No specific fashion tags matched the dictionary rules.")

In [ ]:
sample_text_2 = "I bought this lovely cotton and linen blend maxi dress. It is beautifully oversized!"
run_approach_two_ner(sample_text_2)

29. Final Engine Pipeline

In [ ]:
def deep_fashion_search_final(query, k=2):
    # FAISS Vector Search matching
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    print(f"COMPLETE AI ENGINE PIPELINE FOR QUERY: '{query.upper()}'")
    print("-" * 60 + "\n")

    for score, idx in zip(D[0], I[0]):
        title = df.iloc[idx]["Title"]
        review = df.iloc[idx]["Review Text"]

        print(f"Similarity Score: {round(float(score), 2)}")
        print(f"Review Title: {title}")

        # Review Summarization using BART
        summary_result = summarizer(str(review), max_length=50, min_length=20, do_sample=False)
        print(f"AI Summary: {summary_result[0]['summary_text']}\n")

        # Run both stable NER approaches
        run_approach_one_ner(str(review))
        run_approach_two_ner(str(review))

        print("=" * 60 + "\n")

30. Testing complete engine

In [ ]:
deep_fashion_search_final("party wear, sparkle and lightweight dress for a hot sunny beach vacation", k=2)

In [ ]:
deep_fashion_search_final("cotton shirt for daily wear and office wear", k=5)

31. Interactive AI Fashion Search Prompt

In [ ]:
user_query = input("What kind of outfit are you looking for today? ")
deep_fashion_search_final(user_query, k=2)